In [0]:
from pyspark.sql import functions as F

LANDING = "/Volumes/electrocasa/bronze/landing"

RUTA_VENTAS = f"{LANDING}/ventas_sucursales.csv"
RUTA_PRODUCTOS = f"{LANDING}/catalogo_productos.json"
RUTA_EMPLEADOS = f"{LANDING}/empleados_rrhh.csv"
RUTA_RESENAS = f"{LANDING}/resenas_clientes.json"
RUTA_DEVOLUCIONES = f"{LANDING}/devoluciones.csv"

print("Rutas configuradas correctamente")

### Ventas Sucursales

In [0]:
df_ventas = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(RUTA_VENTAS)
)

print(f"Total registros ventas: {df_ventas.count()}")
df_ventas.printSchema()

In [0]:
display(df_ventas.limit(20))

In [0]:
display(
    df_ventas.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_ventas.columns
    ])
)

In [0]:
display(
    df_ventas
    .groupBy("metodo_pago")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_ventas.filter(
        F.col("monto_total").isNull() |
        (F.col("monto_total") <= 0)
    )
)

In [0]:
display(
    df_ventas.filter(
        F.col("cantidad").isNull() |
        (F.col("cantidad") <= 0)
    )
)

In [0]:
display(
    df_ventas
    .groupBy("venta_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

### Catalogo de Productos

In [0]:
df_productos = (
    spark.read
    .option("multiLine", "true")
    .json(RUTA_PRODUCTOS)
)

print(f"Total registros productos: {df_productos.count()}")
df_productos.printSchema()

In [0]:
display(df_productos.limit(20))

In [0]:
display(
    df_productos
    .groupBy("categoria")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_productos.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_productos.columns
    ])
)

In [0]:
df_productos_explora = df_productos.withColumn(
    "precio_numerico",
    F.regexp_replace(
        F.col("precio_lista").cast("string"),
        r"[^0-9.\-]",
        ""
    ).cast("double")
)

display(
    df_productos_explora.filter(
        F.col("precio_numerico").isNull() |
        (F.col("precio_numerico") <= 0)
    )
)

In [0]:
display(
    df_productos
    .groupBy("producto_id")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("precio_lista").alias("precios_distintos")
    )
    .filter(F.col("registros") > 1)
    .orderBy(F.desc("registros"))
)

### Empleados de RRHH

In [0]:
df_empleados = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(RUTA_EMPLEADOS)
)

print(f"Total registros empleados: {df_empleados.count()}")
df_empleados.printSchema()

In [0]:
display(df_empleados.limit(20))

In [0]:
display(
    df_empleados.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_empleados.columns
    ])
)

In [0]:
display(
    df_empleados
    .groupBy("tipo_evento")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_empleados
    .groupBy("tipo_evento")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_empleados
    .groupBy("id_empleado")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_empleados.filter(
        F.col("dni").isNull()
    )
)

In [0]:
display(
    df_empleados
    .groupBy("dni")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("id_empleado").alias("empleados_distintos")
    )
    .filter(
        F.col("dni").isNotNull() &
        (F.col("empleados_distintos") > 1)
    )
    .orderBy(F.desc("empleados_distintos"))
)

In [0]:
empleados_con_historial = (
    df_empleados
    .groupBy("id_empleado")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

display(empleados_con_historial.limit(20))

In [0]:
display(
    df_empleados
    .filter(F.col("id_empleado") == "E01673")
    .orderBy("fecha_evento")
)

### Reseña de clientes

In [0]:
print(
    dbutils.fs.head(
        RUTA_RESENAS,
        2000
    )
)

In [0]:
df_resenas = (
    spark.read
    .option("multiLine", "true")
    .json(RUTA_RESENAS)
)

print(f"Total registros reseñas: {df_resenas.count()}")
df_resenas.printSchema()

In [0]:
display(df_resenas.limit(20))

In [0]:
display(
    df_resenas.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_resenas.columns
    ])
)

In [0]:
display(
    df_resenas
    .groupBy("calificacion")
    .count()
    .orderBy("calificacion")
)

In [0]:
display(
    df_resenas.filter(
        F.col("calificacion").isNull() |
        (F.col("calificacion") < 1) |
        (F.col("calificacion") > 5)
    )
)

In [0]:
display(
    df_resenas
    .groupBy("resena_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_resenas
    .select(
        "resena_id",
        "calificacion",
        F.size("tags").alias("cantidad_tags"),
        "tags"
    )
    .orderBy(F.desc("cantidad_tags"))
    .limit(50)
)

In [0]:
display(
    df_resenas
    .select(F.explode_outer("tags").alias("tag"))
    .groupBy("tag")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_resenas
    .select(
        "resena_id",
        F.size("respuestas").alias("cantidad_respuestas"),
        "respuestas"
    )
    .filter(F.col("cantidad_respuestas") > 0)
    .orderBy(F.desc("cantidad_respuestas"))
    .limit(50)
)

### Devoluciones

In [0]:
df_devoluciones = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(RUTA_DEVOLUCIONES)
)

print(f"Total registros devoluciones: {df_devoluciones.count()}")
df_devoluciones.printSchema()

In [0]:
display(df_devoluciones.limit(20))

In [0]:
display(
    df_devoluciones.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_devoluciones.columns
    ])
)

In [0]:
display(
    df_devoluciones
    .groupBy("motivo")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_devoluciones.filter(
        F.col("monto_reembolso").isNull() |
        (F.col("monto_reembolso") < 0)
    )
)

In [0]:
display(
    df_devoluciones
    .groupBy("devolucion_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_devoluciones
    .groupBy("pedido_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

In [0]:
print(f"Total registros devoluciones: {df_devoluciones.count()}")

### Tracking  de envios(Azure SQL)

In [0]:
df_tracking = spark.table("electrocasa_sql.dbo.trackingenvios")

print(f"Total registros tracking: {df_tracking.count()}")
df_tracking.printSchema()

In [0]:
display(df_tracking.limit(20))

In [0]:
display(
    df_tracking.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_tracking.columns
    ])
)

In [0]:
display(
    df_tracking
    .groupBy("estado_entrega")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_tracking
    .groupBy("courier")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_tracking
    .groupBy("tracking_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_tracking
    .groupBy("pedido_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)